In [11]:
# ============================================================
# Systematic Literature Review Processing Pipeline
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import ast

from scripts.data_preprocess import bib_to_csv

# ============================================================
# Configuration
# ============================================================

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
FINAL_DIR = Path("data/final")
FIG_DIR = Path("data/figures")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")

# ============================================================
# Global Matplotlib Styling
# ============================================================

plt.rcParams.update({

    # Figure sizing
    "figure.figsize": (11, 6),

    # Fonts
    "font.size": 11,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,

    # Better spacing
    "axes.titlepad": 24,   # more space between title and plot
    "axes.labelpad": 14,   # more space between axis labels and axes

    # Grid
    "grid.alpha": 0.3,
    "grid.linestyle": "--",

    # Lines
    "lines.linewidth": 2,

    # Saving
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

YEAR_THRESHOLD = 2022

# ============================================================
# Utility Functions
# ============================================================

def normalize_doi(series):

    return (
        series.astype(str)
        .str.lower()
        .str.strip()
        .str.replace("https://doi.org/", "", regex=False)
        .str.replace("http://doi.org/", "", regex=False)
        .replace("nan", np.nan)
    )


def load_with_source(folder_path, source_name):

    folder = Path(folder_path)

    dfs = []

    for file in folder.glob("*.csv"):

        df = pd.read_csv(file)
        df["source"] = source_name

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


def filter_publication_type(df, column_name):

    pattern = re.compile(
        r"conference|journal|article|proceedings",
        re.IGNORECASE
    )

    cleaned_col = (
        df[column_name]
        .astype(str)
        .str.strip()
    )

    return df[
        cleaned_col.str.contains(pattern, na=False)
    ]


def filter_title_abstract(df, column_names):

    fall_pattern = r"\bfall[\s\-]detection\b"
    ps_pattern = r"\b(privacy|security)\b"

    mask = pd.Series(False, index=df.index)

    for col in column_names:

        text = (
            df[col]
            .astype(str)
            .str.strip()
        )

        fall_match = text.str.contains(
            fall_pattern,
            case=False,
            regex=True,
            na=False
        )

        ps_match = text.str.contains(
            ps_pattern,
            case=False,
            regex=True,
            na=False
        )

        mask |= (fall_match & ps_match)

    return df[mask]


def save_plot(name):

    plt.tight_layout(
        pad=2.4,
        w_pad=1.8,
        h_pad=1.8
    )

    plt.savefig(
        FIG_DIR / f"{name}.pdf",
        bbox_inches="tight",
        pad_inches=0.35
    )

    plt.close()


def keywords_to_list(s):

    if s is None:
        return []

    if isinstance(s, float) and pd.isna(s):
        return []

    if isinstance(s, (list, np.ndarray)):

        cleaned = []

        for item in s:

            if item is None:
                continue

            cleaned.extend([
                x.strip()
                for x in re.split(r"[;,]", str(item))
                if x.strip()
            ])

        return cleaned

    s = str(s).strip()

    if s == "":
        return []

    try:

        parsed = ast.literal_eval(s)

        if isinstance(parsed, list):

            cleaned = []

            for item in parsed:

                cleaned.extend([
                    x.strip()
                    for x in re.split(r"[;,]", str(item))
                    if x.strip()
                ])

            return cleaned

    except:
        pass

    return [
        x.strip()
        for x in re.split(r"[;,]", s)
        if x.strip()
    ]


# ============================================================
# Count Logging
# ============================================================

filter_log = []


def log_counts(step, acm, ieee, sd):

    filter_log.append({
        "Step": step,
        "ACM": len(acm),
        "IEEE": len(ieee),
        "ScienceDirect": len(sd),
        "Total": len(acm) + len(ieee) + len(sd)
    })


# ============================================================
# Convert BIB files to CSV
# ============================================================

acm_paths = [
    "data/raw/ACM/acm_query_abstract_2026-03-02.bib",
    "data/raw/ACM/acm_query_title_2026-03-02.bib"
]

bib_to_csv(
    acm_paths,
    "data/processed/acm_query.csv"
)

bib_to_csv(
    "data/raw/ScienceDirect/sd_query_2026-03-02.bib",
    "data/processed/sd_query.csv"
)

# ============================================================
# Load Data
# ============================================================

acm = pd.read_csv(
    "data/processed/acm_query.csv"
)

acm["source"] = "ACM"

ieee = load_with_source(
    "data/raw/IEEE",
    "IEEE"
)

sd = pd.read_csv(
    "data/processed/sd_query.csv"
)

sd["source"] = "ScienceDirect"

# ============================================================
# Initial Counts
# ============================================================

log_counts("Raw records", acm, ieee, sd)

# ============================================================
# Normalize DOI
# ============================================================

acm["doi"] = normalize_doi(acm["doi"])
ieee["DOI"] = normalize_doi(ieee["DOI"])
sd["doi"] = normalize_doi(sd["doi"])

# ============================================================
# Remove Duplicates Within Databases
# ============================================================

acm.drop_duplicates(
    subset="doi",
    inplace=True
)

ieee.drop_duplicates(
    subset="DOI",
    inplace=True
)

sd.drop_duplicates(
    subset="doi",
    inplace=True
)

log_counts(
    "Duplicate removal",
    acm,
    ieee,
    sd
)

# ============================================================
# Filter Publication Types
# ============================================================

acm = filter_publication_type(
    acm,
    "Item Type"
)

ieee = filter_publication_type(
    ieee,
    "Document Identifier"
)

sd = filter_publication_type(
    sd,
    "Item Type"
)

log_counts(
    "Publication type filtering",
    acm,
    ieee,
    sd
)

# ============================================================
# Title / Abstract Filtering
# ============================================================

acm = filter_title_abstract(
    acm,
    ["title", "abstract"]
)

ieee = filter_title_abstract(
    ieee,
    ["Document Title", "Abstract"]
)

sd = filter_title_abstract(
    sd,
    ["title", "abstract"]
)

log_counts(
    "Title/abstract filtering",
    acm,
    ieee,
    sd
)

# ============================================================
# Save Intermediate Files
# ============================================================

acm.to_csv(
    FINAL_DIR / "acm_filtered.csv",
    index=False
)

ieee.to_csv(
    FINAL_DIR / "ieee_filtered.csv",
    index=False
)

sd.to_csv(
    FINAL_DIR / "sd_filtered.csv",
    index=False
)

# ============================================================
# Standardize Columns
# ============================================================

acm = acm.rename(columns={
    "year": "Year",
    "keywords": "Keywords"
})

ieee = ieee.rename(columns={
    "Publication Year": "Year",
    "Author Keywords": "Keywords",
    "DOI": "doi"
})

sd = sd.rename(columns={
    "year": "Year",
    "keywords": "Keywords"
})

# ============================================================
# Merge IEEE Terms
# ============================================================

if "IEEE Terms" in ieee.columns:

    ieee["IEEE Terms"] = (
        ieee["IEEE Terms"]
        .apply(keywords_to_list)
    )

    ieee["Keywords"] = (
        ieee["Keywords"]
        .apply(keywords_to_list)
    )

    ieee["Keywords"] = (
        ieee["Keywords"] +
        ieee["IEEE Terms"]
    )

# ============================================================
# Normalize Keywords
# ============================================================

for df in [acm, ieee, sd]:

    if "Keywords" in df.columns:

        df["Keywords"] = (
            df["Keywords"]
            .apply(keywords_to_list)
        )

# ============================================================
# Merge All Datasets
# ============================================================

all_data = pd.concat(
    [acm, ieee, sd],
    ignore_index=True
)

# ============================================================
# Remove Cross-Database Duplicates
# ============================================================

before = len(all_data)

all_data.drop_duplicates(
    subset="doi",
    inplace=True
)

after = len(all_data)

print(
    f"Removed {before - after} "
    f"cross-database duplicates"
)

# ============================================================
# Convert Year Column
# ============================================================

all_data["Year"] = pd.to_numeric(
    all_data["Year"],
    errors="coerce"
)

all_data = all_data.dropna(
    subset=["Year"]
)

all_data["Year"] = (
    all_data["Year"]
    .astype(int)
)

# ============================================================
# FIGURE 1
# Publications Per Year
# ============================================================

year_counts = (
    all_data
    .groupby("Year")
    .size()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(
    year_counts.index,
    year_counts.values
)

# Add counts above bars
for bar in bars:

    height = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.3,
        f"{int(height)}",
        ha='center',
        va='bottom'
    )

ax.set_xlabel("Publication Year")
ax.set_ylabel("Number of Studies")
ax.set_title("Publications per Year")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

save_plot("publications_per_year")

# ============================================================
# FIGURE 2
# Cumulative Publications (Bar Plot)
# ============================================================

cumulative_counts = (
    year_counts.cumsum()
)

plt.figure(figsize=(11, 6))

bars = plt.bar(
    cumulative_counts.index.astype(str),
    cumulative_counts.values,
    edgecolor="black",
    linewidth=0.8
)

# Add value labels
for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 1,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.xlabel(
    "Publication Year",
    fontsize=12
)

plt.ylabel(
    "Cumulative Publications",
    fontsize=12
)

plt.title(
    "Cumulative Publications Over Time",
    fontsize=14,
    weight="bold"
)

plt.xticks(rotation=45)

plt.ylim(
    0,
    cumulative_counts.max() * 1.12
)

save_plot("cumulative_publications")

# ============================================================
# IEEE STUDIES BEFORE 2022
# ============================================================

ieee_before_2022 = all_data[
    (all_data["source"] == "IEEE") &
    (all_data["Year"] < 2022)
]

print(
    f"IEEE studies published before 2022: "
    f"{len(ieee_before_2022)}"
)

# ============================================================
# FILTER BY PUBLICATION YEAR
# ============================================================

before_year_filter = len(all_data)

all_data = all_data[
    all_data["Year"] >= YEAR_THRESHOLD
].copy()

after_year_filter = len(all_data)

print(
    f"Removed "
    f"{before_year_filter - after_year_filter} "
    f"studies published before "
    f"{YEAR_THRESHOLD}"
)

# ============================================================
# Log Year Filtering
# ============================================================

filter_log.append({
    "Step": f"Year filtering ({YEAR_THRESHOLD}+)",
    "ACM": len(
        all_data[
            all_data["source"] == "ACM"
        ]
    ),
    "IEEE": len(
        all_data[
            all_data["source"] == "IEEE"
        ]
    ),
    "ScienceDirect": len(
        all_data[
            all_data["source"] == "ScienceDirect"
        ]
    ),
    "Total": len(all_data)
})

# ============================================================
# FIGURE 3
# Publications Per Year After Filtering
# ============================================================

filtered_year_counts = (
    all_data
    .groupby("Year")
    .size()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(
    filtered_year_counts.index,
    filtered_year_counts.values
)

for bar in bars:

    height = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.2,
        f"{int(height)}",
        ha='center'
    )

ax.set_xlabel("Publication Year")
ax.set_ylabel("Number of Studies")

ax.set_title(
    f"Publications per Year After {YEAR_THRESHOLD}+ Filtering"
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

save_plot("publications_per_year_filtered")

# ============================================================
# FIGURE 4
# Database Distribution
# ============================================================

source_counts = (
    all_data["source"]
    .value_counts()
)

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    source_counts.index,
    source_counts.values
)

# Add counts
for bar in bars:

    width = bar.get_width()

    ax.text(
        width + 0.5,
        bar.get_y() + bar.get_height()/2,
        f"{int(width)}",
        va='center'
    )

ax.set_xlabel("Number of Studies")
ax.set_ylabel("Database")

ax.set_title(
    "Distribution of Studies Across Databases"
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

save_plot("database_distribution")

# ============================================================
# Create Count Table
# ============================================================

count_df = pd.DataFrame(filter_log)

count_df.to_csv(
    FINAL_DIR / "filtering_counts.csv",
    index=False
)

print(count_df)

# ============================================================
# FIGURE 5
# Filtering Pipeline
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

steps = count_df["Step"]
totals = count_df["Total"]

ax.plot(
    steps,
    totals,
    marker="o",
    linewidth=2
)

# Add count labels
for x, y in zip(steps, totals):

    ax.text(
        x,
        y + 5,
        str(y),
        ha='center'
    )

ax.set_ylabel("Number of Records")

ax.set_title(
    "Study Selection and Filtering Process"
)

plt.xticks(rotation=20)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

save_plot("filtering_pipeline")

# ============================================================
# FIGURE 6
# Top Keywords
# ============================================================

keywords_exploded = (
    all_data
    .explode("Keywords")
)

keywords_exploded["Keywords"] = (
    keywords_exploded["Keywords"]
    .astype(str)
    .str.lower()
    .str.strip()
)

keyword_counts = (
    keywords_exploded["Keywords"]
    .value_counts()
    .head(20)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(
    keyword_counts.index,
    keyword_counts.values
)

# Add labels
for bar in bars:

    width = bar.get_width()

    ax.text(
        width + 0.2,
        bar.get_y() + bar.get_height()/2,
        f"{int(width)}",
        va='center'
    )

ax.set_xlabel("Frequency")
ax.set_ylabel("Keyword")

ax.set_title("Most Frequent Keywords")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

save_plot("top_keywords")

# ============================================================
# Save Final Dataset
# ============================================================

all_data.to_csv(
    FINAL_DIR / "all_filtered_studies.csv",
    index=False
)

# ============================================================
# PRISMA Summary
# ============================================================

prisma = {
    "Records identified":
        count_df.iloc[0]["Total"],

    "After duplicates removed":
        count_df.iloc[1]["Total"],

    "After publication filtering":
        count_df.iloc[2]["Total"],

    "After title/abstract filtering":
        count_df.iloc[3]["Total"],

    f"After year filtering ({YEAR_THRESHOLD}+)":
        count_df.iloc[4]["Total"]
}

print("\nPRISMA SUMMARY")
print(prisma)

# ============================================================
# Finished
# ============================================================

print("\nPipeline completed successfully.")

/var/folders/vl/562b88_13rd3bp112l2864c00000gn/T/ipykernel_92621/1755147838.py:136: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ps_match = text.str.contains(
/var/folders/vl/562b88_13rd3bp112l2864c00000gn/T/ipykernel_92621/1755147838.py:136: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ps_match = text.str.contains(
/var/folders/vl/562b88_13rd3bp112l2864c00000gn/T/ipykernel_92621/1755147838.py:136: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ps_match = text.str.contains(
/var/folders/vl/562b88_13rd3bp112l2864c00000gn/T/ipykernel_92621/1755147838.py:136: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ps_match = text.str.contains(
/var

Removed 0 cross-database duplicates
IEEE studies published before 2022: 128
Removed 152 studies published before 2022
                         Step  ACM  IEEE  ScienceDirect  Total
0                 Raw records   78   337             49    464
1           Duplicate removal   72   315             49    436
2  Publication type filtering   71   315             49    435
3    Title/abstract filtering   29   315             38    382
4      Year filtering (2022+)   21   187             22    230

PRISMA SUMMARY
{'Records identified': np.int64(464), 'After duplicates removed': np.int64(436), 'After publication filtering': np.int64(435), 'After title/abstract filtering': np.int64(382), 'After year filtering (2022+)': np.int64(230)}

Pipeline completed successfully.
